# 09 - Final result aggregation

Aggregates CSV files produced by notebooks 05 and 08.

In [ ]:
import exp_common as E
import pandas as pd

base_raw = E.RESULTS_DIR/"baselines"/"baseline_metrics_raw.csv"
merged_raw = E.RESULTS_DIR/"merged"/"merged_metrics_raw.csv"
frames=[]
if base_raw.exists():
    b=pd.read_csv(base_raw); b["group"]="baseline"; frames.append(b)
if merged_raw.exists():
    m=pd.read_csv(merged_raw); m["group"]="merged"; frames.append(m)
if not frames:
    raise FileNotFoundError("Run 05 and 08 first")
all_df=pd.concat(frames,ignore_index=True)
all_df.to_csv(E.RESULTS_DIR/"all_metrics_raw.csv",index=False)

num_cols=[c for c in all_df.columns if c not in ["model","seed","group"] and pd.api.types.is_numeric_dtype(all_df[c])]
summary=all_df.groupby(["group","model"])[num_cols].agg(["mean","std","count"])
summary.to_csv(E.RESULTS_DIR/"all_metrics_mean_std.csv")

main_cols=["logs_accuracy",
           "logs_precision_anomaly",
           "logs_recall_anomaly",
           "logs_f1_anomaly",
           "logs_coverage_strict",
           "logs_off_format_rate",
           "logs_pred_anomaly_rate",
           "logs_keyword_accuracy_sensitivity",
           "qa_semantic_similarity_mean",
           "qa_llm_judge_mean_0_5",
           "qa_judge_invalid_rate"]
rows=[]
for (group,model),sub in all_df.groupby(["group","model"]):
    row={"group":group,"model":model,"n_seeds":sub["seed"].nunique()}
    for c in main_cols:
        if c in sub.columns:
            row[c+"__mean"]=sub[c].mean()
            row[c+"__std"]=sub[c].std(ddof=1) if len(sub)>1 else float("nan")
    rows.append(row)
report=pd.DataFrame(rows).sort_values(["group","model"])
report.to_csv(E.RESULTS_DIR/"main_report_camera_ready.csv",index=False)
report